# 24. 选择、筛选与排序

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 3 / 10 步：读懂并定位表中信息**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** Series与DataFrame  →  **本章任务：** 选择、筛选与排序  →  **下一步：** 行列操作与类型转换
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

拿到一张真实表格后，第一步往往就是挑出关心的行、筛掉无关的记录、再按高低排个顺序。



## 本章目标

学完本章，你将能够：

- **理解**：理解布尔筛选、loc/iloc 与 sort_values。
- **操作**：能按条件筛选、定位、排序 DataFrame。
- **迁移**：能从订单表筛出满足条件（如大额、指定渠道）的记录并排序。


## 24.1 核心概念

**背景引入**：拿到一张真实表格后，第一步往往就是挑出关心的行、筛掉无关的记录、再按高低排个顺序。无论是几个人的订单，还是几十万行的销售明细，选择、筛选与排序都是数据分析里最常用、最基础的一组动作——把它们练熟，后面的分组、汇总和可视化才站得稳。

- loc按标签选择且切片包含终点，iloc按位置选择且右端不包含。
- 多个布尔条件必须分别加括号。
- 筛选前先检查缺失值和数据类型。

> **直观类比**：loc 像“按门牌号找人”，1～5 号这几户都算（含终点）；iloc 像“按第几户数”，从 0 号数起、且“数到 5”只到第 4 户（右端不含）。认准“找标签”还是“数位置”，切片边界就不会混。


## 24.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| loc | `pd.DataFrame()`、`loc[['A1', 'A3']` | loc使用行标签和列标签，切片通常包含结束标签。 | loc和iloc切片边界规则混淆 |
| iloc | `pd.DataFrame()`、`iloc[:2, 1]` | iloc使用整数位置，切片右端不包含。 | 多个条件之间漏写括号 |
| 布尔条件筛选 | `pd.DataFrame()`、`orders[(orders['amount']`、`orders['region']` | 多个条件需要分别加括号，再使用&或\|组合。 | 排序后仍使用旧的位置含义 |
| query() | `pd.DataFrame()`、`orders.query()` | query适合写成接近自然语言的列条件。 | loc和iloc切片边界规则混淆 |
| sort_values() 与 nlargest() | `pd.DataFrame()`、`orders.sort_values()`、`orders.nlargest()` | 排序用于完整排名，nlargest适合快速取得Top N。 | 多个条件之间漏写括号 |
| isin() 与 between() | `pd.DataFrame()`、`.isin()`、`.between()`、`orders[orders['region']` | 这两个方法适合表达集合筛选和区间筛选。 | 排序后仍使用旧的位置含义 |


## 24.3 示例 1：loc与iloc

**背景引入**：订单表里有的订单用编号 A1、A2 当行名，有的就看第几行。想精准抽出“A2、A4 两单的地区和金额”，总不能在 Excel 里一行行点。loc 跟 iloc 就是两把“定位镊子”，一个认标签、一个认位置。

**讲解**：`loc` 用**行/列标签**挑数据，`iloc` 用**整数位置**挑数据，两把镊子握法不同。

- `orders.loc[["A2","A4"], ["region","amount"]]`：先按行标签选两行，再按列名只留两列，无关数据不进后续计算；
- `orders.iloc[:3, [0, 2]]`：`iloc` 前三行、第 0 和第 2 列，注意**右端不包含**、列按整数下标；
- 列多了用 `loc` 显式只挑需要的列，能减少后续算错的范围；
- **口诀**：loc 按“名字”loc，iloc 按“座位号”，切片右端记住不取。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华南", "华东", "华北", "华南"],
        "channel": ["线上", "线下", "线上", "线上", "线下"],
        "amount": [320, 880, 460, 1250, 720],
    },
    index=["A1", "A2", "A3", "A4", "A5"],
)
print(orders.loc[["A2", "A4"], ["region", "amount"]])
print(orders.iloc[:3, [0, 2]])


## 24.4 示例 2：条件筛选与query

**背景引入**：领导只想要“金额在 500 以上、且走线上”的订单，手动在几百行里翻不现实。把“地域＋渠道＋金额”这些条件写成表达式，Pandas 就能一次把符合条件的行全筛出来——这就是分析师每天干得最多的活。

**讲解**：布尔筛选用 `orders[条件]`，条件各自加括号再组合；`query` 则把条件写成更像人话的字符串。

- `(orders["amount"] >= 500) & (orders["channel"] == "线上")`：多个条件**分别加括号**再 `&`，漏括号最容易出 bug；
- `orders.query("amount >= 500 and region != '华北'")`：query 直接把列条件写成字符串，读起来像自然语言；
- 布尔掩码适合复杂、动态拼出来的逻辑，query 适合一眼能读懂的列条件，各有所长；
- **口诀**：括号给每个条件，& 是“且”｜是“或”，想读得顺就用 query。


In [ ]:
selected = orders[(orders["amount"] >= 500) & (orders["channel"] == "线上")]
queried = orders.query("amount >= 500 and region != '华北'")
print(selected)
print(queried)


## 24.5 示例 3：排序与Top N

**背景引入**：洗完数据，领导最常问的一句话是“卖得最好的前三单是谁？”把订单按金额排个名，看看头部客户、头部地区贡献了多少——排序和取前几，是排在筛选之后的第二高频动作。

**讲解**：`sort_values` 给整张表排名，`nlargest` 直接腾出前几名，选择取决于你要“全排名”还是“只看头名”。

- `orders.sort_values(["amount","region"], ascending=[False,True])`：先用金额降序，金额相同再按地区升序，多列排序方向要分别给；
- `orders.nlargest(3, "amount")`：只要金额最大的前 3 行，一步到位，不用先排再 `.head(3)`；
- 记得排完序是**新的顺序**，再用旧的位置含义去取行会拿错数据；
- **口诀**：要完整排名用 sort_values，只争头部用 nlargest，排完就别拿旧位置说事。


In [ ]:
ranked = orders.sort_values(["amount", "region"], ascending=[False, True])
top_three = orders.nlargest(3, "amount")
print(ranked)
print("Top 3:\n", top_three)


## 24.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# loc
# loc使用行标签和列标签，切片通常包含结束标签。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华南", "华北"], "amount": [320, 880, 460]},
    index=["A1", "A2", "A3"],
)
print(orders.loc[["A1", "A3"], ["region", "amount"]])


In [ ]:
# iloc
# iloc使用整数位置，切片右端不包含。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华南", "华北"], "amount": [320, 880, 460]}
)
print(orders.iloc[:2, 1])


In [ ]:
# 布尔条件筛选
# 多个条件需要分别加括号，再使用&或|组合。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华南", "华北"], "amount": [320, 880, 460]}
)
selected = orders[(orders["amount"] >= 400) & (orders["region"] != "华北")]
print(selected)


In [ ]:
# query()
# query适合写成接近自然语言的列条件。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华南", "华北"], "amount": [320, 880, 460]}
)
print(orders.query("amount >= 400 and region != '华北'"))


In [ ]:
# sort_values() 与 nlargest()
# 排序用于完整排名，nlargest适合快速取得Top N。
import pandas as pd

orders = pd.DataFrame({"id": ["A1", "A2", "A3"], "amount": [320, 880, 460]})
print(orders.sort_values("amount", ascending=False))
print(orders.nlargest(2, "amount"))


In [ ]:
# isin() 与 between()
# 这两个方法适合表达集合筛选和区间筛选。
import pandas as pd

orders = pd.DataFrame(
    {"region": ["华东", "华南", "华北"], "amount": [320, 880, 460]}
)
print(orders[orders["region"].isin(["华东", "华南"])])
print(orders[orders["amount"].between(300, 500)])


**练一练 20.6**：给定订单表 orders，完成三件事：先用 `loc` 按行标签选出第 1、3 行（只保留 id 与 amount 两列）；再用布尔条件筛出 amount 大于等于 500 的线上订单（两个条件都要加括号）；最后按 amount 从大到小排序并取最大的前 2 个订单。


In [ ]:
# 请在下方填写代码
import pandas as pd

# 第 1 步：用 loc 选出第 1 行与第 3 行（行标签 r1、r3），只保留 id 与 amount 两列
# 第 2 步：筛选出 amount 大于等于 500 的线上订单（两个条件都要加括号）
# 第 3 步：按 amount 降序排序，再取最大的前 2 个订单


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "id": ["A1", "A2", "A3"],
        "channel": ["线上", "线下", "线上"],
        "amount": [320, 880, 920],
    },
    index=["r1", "r2", "r3"],
)

# 第 1 步：loc 按行标签选 r1、r3，只保留 id 与 amount 两列
step1 = orders.loc[["r1", "r3"], ["id", "amount"]]

# 第 2 步：布尔条件筛选，两个条件分别加括号再 & 组合
step2 = orders[(orders["amount"] >= 500) & (orders["channel"] == "线上")]

# 第 3 步：sort_values 降序后取前 2 个订单
step3 = orders.sort_values("amount", ascending=False).head(2)


## 24.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)
large_orders.head()


In [ ]:
completed = large_orders.query("status == '完成' and sales > 0")
high_value = completed.loc[
    completed["sales"] >= completed["sales"].quantile(0.99),
    ["order_id", "stock_code", "description", "country", "sales"],
].sort_values("sales", ascending=False)
print(f"Top 1% 高价值订单：{len(high_value):,} 条")
display(high_value.head(10))


## 24.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 24.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 24.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 24.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 24.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 24.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 24.11 易错点提醒

- loc和iloc切片边界规则混淆
- 多个条件之间漏写括号
- 排序后仍使用旧的位置含义


## 24.12 练习与作业

1. 筛选华东或华南订单
2. 金额不低于400
3. 按金额降序返回前三条

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 24.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“筛选华东或华南订单”。
2. **独立完成**：不复制示例代码，完成“金额不低于400”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“按金额降序返回前三条”，用一两句话说明你修改了什么。

### 24.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 24.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

# TODO: 筛选华东或华南且金额不低于400的订单
# TODO: 按金额降序并返回前三条
# TODO：请在下方完成 —— 20.13 练习与作业 1. 筛选华东或华南订单 2. 金额不低于400 3. 按金额降序返回前三条 提交前检查：代码可


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "华东", "西南"],
        "amount": [320, 880, 460, 1250, 720],
        "status": ["完成", "完成", "取消", "完成", "完成"],
    }
)
result = (
    orders[orders["region"].isin(["华东", "华南"]) & (orders["amount"] >= 400)]
    .sort_values("amount", ascending=False)
    .head(3)
)
print(result)


## 24.14 小结

使用loc、iloc、条件表达式、query和排序准确定位数据。

**迁移思考**：

1. 如果需要筛选"金额大于500或地区为华东"的订单，布尔表达式应该如何写？
2. 为什么 loc 切片包含终点而 iloc 切片不包含？这种差异在什么场景下容易出错？



### 24.14.1 你已经掌握

- 按标签与位置选择
- 组合多条件筛选
- 使用query表达条件
- 按一列或多列排序



### 24.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 24.14.3 需要注意

- loc和iloc切片边界规则混淆
- 多个条件之间漏写括号
- 排序后仍使用旧的位置含义



### 24.14.4 完成检查

- [ ] 能够按标签与位置选择
- [ ] 能够组合多条件筛选
- [ ] 能够使用query表达条件
- [ ] 能够按一列或多列排序



### 24.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

